# Food Delivery Sales & Operations Analytics
Synthetic portfolio project using SQLite, SQL, Python and pandas.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

conn = sqlite3.connect('food_delivery.db')
conn.execute('PRAGMA foreign_keys = ON')
conn.executescript(Path('sql/schema.sql').read_text())
for table in ['customers', 'restaurants', 'orders', 'restaurant_tags']:
    pd.read_csv(Path('data') / f'{table}.csv').to_sql(table, conn, if_exists='append', index=False)
print('Database loaded')


## Delivered revenue by cuisine


In [ ]:
pd.read_sql_query('''
SELECT r.cuisine, COUNT(*) AS delivered_orders, ROUND(SUM(o.amount), 2) AS revenue
FROM orders o JOIN restaurants r ON r.restaurant_id = o.restaurant_id
WHERE o.status = 'delivered'
GROUP BY r.cuisine ORDER BY revenue DESC
''', conn)


## Daily delivered revenue


In [ ]:
pd.read_sql_query('''
SELECT order_date, COUNT(*) AS delivered_orders, ROUND(SUM(amount), 2) AS revenue
FROM orders WHERE status = 'delivered'
GROUP BY order_date ORDER BY order_date
''', conn)


## Rank cuisines using a CTE and window function


In [ ]:
pd.read_sql_query('''
WITH cuisine_revenue AS (
  SELECT r.cuisine, SUM(o.amount) AS revenue
  FROM orders o JOIN restaurants r ON r.restaurant_id = o.restaurant_id
  WHERE o.status = 'delivered' GROUP BY r.cuisine
)
SELECT cuisine, revenue, DENSE_RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
FROM cuisine_revenue ORDER BY revenue_rank
''', conn)


In [ ]:
conn.close()
